# Gold Layer Analytics Notebook

This notebook transforms cleaned Silver-layer Delta tables into analytics-ready Gold-layer tables for the parcel sorting hub data model.

The Gold layer is responsible for:

- creating fact tables for operational analysis,
- creating helper tables for reusable business logic,
- creating dimension tables for Power BI filtering and slicing,
- calculating KPI flags and process duration metrics,
- preparing data at the correct analytical grain for dashboard reporting.

The notebook follows the Medallion Architecture pattern:

Bronze -> Raw ingested data
Silver -> Cleaned, validated, standardized data
Gold -> Business-ready analytical tables

Only cleaned Silver-layer records are used as inputs for Gold-layer tables.

## Gold Table Design

Gold tables are designed for Power BI reporting and operational KPI analysis.

The tables are split into three groups:

### 1. Helper tables
- Reusable intermediate tables used to simplify fact table logic.
### 2. Fact tables
- Analytical tables containing measurable business events, durations, statuses, and KPI flags.
### 3. Dimension tables
- Small lookup/filter tables used for slicing reports by date, hour, and shift.

# Fact and Helper Tables

# `gold.parcel_scan_process`

### Purpose

This helper table converts multiple scan-event rows into one parcel-level scan process record.

It makes scan logic easier to reuse in the main parcel journey fact table.

### Grain

One row per `parcel_id`.

### Source Tables
- silver.scans
- silver.exceptions

### Business Logic

The table:

- extracts the first timestamp for each scan stage,
- captures scan results for each stage,
- identifies whether a parcel had an exception,
- creates success and failure flags for each scan stage,
- identifies the first failed stage,
- distinguishes between failed, skipped, and exception-related scan outcomes.

### Power BI Usage

**This table should usually not be used directly as a main report table.**

Its main purpose is to support `gold.fact_parcel_journey`.

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.parcel_scan_process 
USING DELTA
AS

-- Reducing all of the scans to one row per parcel_id
WITH parcel_scan_row_unification AS (
    SELECT  
        s.parcel_id,

        MIN(CASE WHEN s.scan_type = 'INBOUND'      THEN s.scan_timestamp END) AS inbound_scan_at,
        MIN(CASE WHEN s.scan_type = 'DIMENSIONING' THEN s.scan_timestamp END) AS dimensioning_scan_at,
        MIN(CASE WHEN s.scan_type = 'DESTINATION'  THEN s.scan_timestamp END) AS destination_scan_at,
        MIN(CASE WHEN s.scan_type = 'LOADING_DOCK' THEN s.scan_timestamp END) AS loading_dock_scan_at,
        MIN(CASE WHEN s.scan_type = 'LOADED'       THEN s.scan_timestamp END) AS loaded_scan_at,

        MIN(CASE WHEN s.scan_type = 'INBOUND'      THEN s.result END) AS inbound_scan_result,
        MIN(CASE WHEN s.scan_type = 'DIMENSIONING' THEN s.result END) AS dimensioning_scan_result,
        MIN(CASE WHEN s.scan_type = 'DESTINATION'  THEN s.result END) AS destination_scan_result,
        MIN(CASE WHEN s.scan_type = 'LOADING_DOCK' THEN s.result END) AS loading_dock_scan_result,
        MIN(CASE WHEN s.scan_type = 'LOADED'       THEN s.result END) AS loaded_scan_result,

        MAX(CASE WHEN e.parcel_id IS NOT NULL THEN 1 ELSE 0 END) AS is_exception

    FROM parcel_sorting_hub.silver.scans s

    LEFT JOIN parcel_sorting_hub.silver.exceptions e
        ON s.parcel_id = e.parcel_id
        
    GROUP BY s.parcel_id
),

-- Business logic flags (ensuring a split between failed scans and skipped scans)
business_flag_system AS (
    SELECT  
        parcel_id,

        inbound_scan_at,
        dimensioning_scan_at,
        destination_scan_at,
        loading_dock_scan_at,
        loaded_scan_at,

    -- SCAN SUCCESS / FAIL CATEGORIZATION FLAG                               
        CASE WHEN inbound_scan_result = 'FAIL'      OR inbound_scan_result      IS NULL THEN 0 ELSE 1 END AS inbound_scan_success,
        CASE WHEN inbound_scan_result = 'FAIL'      OR inbound_scan_result      IS NULL THEN 1 ELSE 0 END AS inbound_scan_failed,

        CASE WHEN dimensioning_scan_result = 'FAIL' OR dimensioning_scan_result IS NULL THEN 0 ELSE 1 END AS dimensioning_scan_success,
        CASE WHEN dimensioning_scan_result = 'FAIL' OR dimensioning_scan_result IS NULL THEN 1 ELSE 0 END AS dimensioning_scan_failed,

        CASE WHEN destination_scan_result = 'FAIL'  OR destination_scan_result  IS NULL THEN 0 ELSE 1 END AS destination_scan_success,
        CASE WHEN destination_scan_result = 'FAIL'  OR destination_scan_result  IS NULL THEN 1 ELSE 0 END AS destination_scan_failed,

        CASE WHEN loading_dock_scan_result = 'FAIL' OR loading_dock_scan_result IS NULL THEN 0 ELSE 1 END AS loading_scan_success,
        CASE WHEN loading_dock_scan_result = 'FAIL' OR loading_dock_scan_result IS NULL THEN 1 ELSE 0 END AS loading_scan_failed,

        CASE WHEN loaded_scan_result = 'FAIL'       OR loaded_scan_result       IS NULL THEN 0 ELSE 1 END AS is_loaded,
        CASE WHEN loaded_scan_result = 'FAIL'       OR loaded_scan_result       IS NULL THEN 1 ELSE 0 END AS not_loaded,

    -- STAGE CATEGORIZATION FLAG
        CASE
            WHEN inbound_scan_result = 'FAIL' THEN 'INBOUND'
            WHEN dimensioning_scan_result = 'FAIL' THEN 'DIMENSIONING'
            WHEN destination_scan_result = 'FAIL' THEN 'DESTINATION'
            WHEN loading_dock_scan_result = 'FAIL' THEN 'LOADING_DOCK'
            WHEN loaded_scan_result = 'FAIL' THEN 'LOADED'
            ELSE 'NO_FAILURE'
        END AS failed_stage,

    -- DISTINCTION BETWEEN FAILED AND SKIPPED SCANS
        CASE 
            WHEN inbound_scan_result IS NULL AND is_exception = 1 THEN 'EXCEPTION' 
            WHEN inbound_scan_result IS NULL AND is_exception = 0 THEN 'SKIPPED'
            ELSE inbound_scan_result 
        END AS inbound_scan_result,

        CASE 
            WHEN dimensioning_scan_result IS NULL AND is_exception = 1 THEN 'EXCEPTION' 
            WHEN dimensioning_scan_result IS NULL AND is_exception = 0 THEN 'SKIPPED'
            ELSE dimensioning_scan_result 
        END AS dimensioning_scan_result,    

        CASE 
            WHEN destination_scan_result IS NULL AND is_exception = 1 THEN 'EXCEPTION' 
            WHEN destination_scan_result IS NULL AND is_exception = 0 THEN 'SKIPPED'
            ELSE destination_scan_result 
        END AS destination_scan_result,    

        CASE 
            WHEN loading_dock_scan_result IS NULL AND is_exception = 1 THEN 'EXCEPTION' 
            WHEN loading_dock_scan_result IS NULL AND is_exception = 0 THEN 'SKIPPED'
            ELSE loading_dock_scan_result 
        END AS loading_dock_scan_result,    

        CASE 
            WHEN loaded_scan_result IS NULL AND is_exception = 1 THEN 'EXCEPTION' 
            WHEN loaded_scan_result IS NULL AND is_exception = 0 THEN 'SKIPPED'
            ELSE loaded_scan_result 
        END AS loaded_scan_result,        

        is_exception

    FROM parcel_scan_row_unification 
)

SELECT
    parcel_id,

    inbound_scan_at,
    dimensioning_scan_at,
    destination_scan_at,
    loading_dock_scan_at,
    loaded_scan_at,
    inbound_scan_result,
    dimensioning_scan_result,
    destination_scan_result,
    loading_dock_scan_result,
    loaded_scan_result,
    failed_stage,

    inbound_scan_success,
    inbound_scan_failed,
    
    dimensioning_scan_success,
    dimensioning_scan_failed,

    destination_scan_success,
    destination_scan_failed,

    loading_scan_success,
    loading_scan_failed,

    is_loaded,
    not_loaded,

    is_exception

FROM business_flag_system

# `gold.fact_parcel_journey`

### Purpose

**This is the main parcel-level fact table.**

It combines parcel information, intake/loading information, scan-process data, sorting results, SLA logic, and process duration metrics into one analytical table.

### Grain

One row per `parcel_id`.

### Source Tables
- silver.parcels
- silver.intake
- silver.loading
- silver.sorting
- gold.parcel_scan_process

### Business Logic
### 
The table includes:

- parcel service type,
- source and destination hub,
- parcel route,
- parcel dimensions and volume,
- parcel size and weight categories,
- inbound and outbound dock identifiers,
- process duration metrics,
- SLA target and SLA result,
- sorting result flags,
- scan-stage timestamps,
- scan-stage result flags,
- exception indicators,
- skipped scan indicators.

### Key Metrics Created

Examples of metrics created in this table:

- registered_to_loaded_minutes
- inbound_to_loaded_minutes
- received_to_inbound_minutes
- inbound_to_dimensioning_minutes
- dimensioning_to_destination_minutes
- destination_to_loading_dock_minutes
- loading_dock_to_loaded_minutes
- is_sla_met
- is_sort_success
- is_sort_rerouted
- is_exception
- is_loaded
- not_loaded

### Power BI Usage

**This should be the main fact table for parcel-level KPIs, including:**

- SLA performance,
- loaded vs not loaded parcels,
- exception rate,
- parcel journey duration,
- parcel size/weight analysis,
- route performance,
- service type performance,
- sorting success rate.****

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.fact_parcel_journey
USING DELTA
AS

SELECT
-- PARCEL INFORMATION
    p.parcel_id,
    p.status AS final_status,
    p.received_at AS parcel_registered_at,
    p.service_type,
    DATE(p.received_at) AS received_date,
    HOUR(p.received_at) AS received_hour,

-- PARCEL ORIGIN AND DESTINATION
    p.source_hub_id,
    p.destination_hub_id,
    CONCAT('FROM ', p.source_hub_id, ' TO ', p.destination_hub_id) AS parcel_travel_route,

-- PARCEL DIMENSIONS
    p.weight_kg,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    ROUND(p.length_cm * p.width_cm * p.height_cm, 2) AS parcel_volume_cm3,

-- PARCEL WEIGHT AND SIZE CATEGORIZATION
    CASE
        WHEN p.weight_kg < 2 THEN 'LIGHT'
        WHEN p.weight_kg < 10 THEN 'MEDIUM'
        ELSE 'HEAVY'
    END AS parcel_weight,

    CASE
        WHEN p.length_cm * p.width_cm * p.height_cm < 10000 THEN 'SMALL'
        WHEN p.length_cm * p.width_cm * p.height_cm < 40000 THEN 'MEDIUM'
        ELSE 'LARGE'
    END AS parcel_size,

-- SOURCE AND DESTINATION HUB IDENTIFIER
    p.delivery_id,
    p.loading_id,

-- INBOUND AND OUTBOUND DOCK IDENTIFIER
    i.dock_id AS inbound_dock_id,
    l.dock_id AS outbound_dock_id,

-- PARCEL PROCESSING TIME
    TIMESTAMPDIFF(MINUTE, i.unload_start, l.close_time) AS unload_start_to_loading_close_minutes,

    CASE 
        WHEN sp.inbound_scan_at IS NOT NULL 
            AND sp.loaded_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, sp.inbound_scan_at, sp.loaded_scan_at)
        ELSE NULL
    END AS inbound_to_loaded_minutes,

    CASE 
        WHEN p.received_at IS NOT NULL 
            AND sp.loaded_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, p.received_at, sp.loaded_scan_at)
        ELSE NULL
    END AS registered_to_loaded_minutes,

-- SLA (FOR PARCEL SERVICE TYPES)
    CASE 
        WHEN p.service_type = 'PRIORITY'
            AND TIMESTAMPDIFF(MINUTE, p.received_at, sp.loaded_scan_at) <= 930 
        THEN 1 

        WHEN p.service_type IN ('LOCKER_24H', 'COURIER_STD') 
            AND TIMESTAMPDIFF(MINUTE, p.received_at, sp.loaded_scan_at) <= 1200 
        THEN 1

        ELSE NULL
    END is_sla_met,

    CASE 
        WHEN p.service_type = 'PRIORITY' THEN 930 
        WHEN p.service_type IN ('LOCKER_24H', 'COURIER_STD') THEN 1200
    END sla_target,

-- SORTING
    s.entry_time AS sort_entry_at,
    s.result AS sort_result,
    CASE WHEN s.result = 'SUCCESS' THEN 1 ELSE 0 END AS is_sort_success,
    CASE WHEN s.result = 'REROUTED' THEN 1 ELSE 0 END AS is_sort_rerouted,

    CASE 
        WHEN s.result = 'SUCCESS' 
        THEN TIMESTAMPDIFF(MINUTE, sp.destination_scan_at, s.entry_time) 
        ELSE NULL 
    END AS destination_to_sort_minutes,
    
-- SCAN TIME DELTA
    CASE 
        WHEN p.received_at IS NOT NULL 
            AND sp.inbound_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, p.received_at, sp.inbound_scan_at)
        ELSE NULL
    END AS received_to_inbound_minutes,

    CASE 
        WHEN sp.inbound_scan_at IS NOT NULL 
            AND sp.dimensioning_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, sp.inbound_scan_at, sp.dimensioning_scan_at)
        ELSE NULL
    END AS inbound_to_dimensioning_minutes,

    CASE 
        WHEN sp.dimensioning_scan_at IS NOT NULL 
            AND sp.destination_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, sp.dimensioning_scan_at, sp.destination_scan_at)
        ELSE NULL
    END AS dimensioning_to_destination_minutes,

    CASE 
        WHEN sp.destination_scan_at IS NOT NULL 
            AND sp.loading_dock_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, sp.destination_scan_at, sp.loading_dock_scan_at)
        ELSE NULL
    END AS destination_to_loading_dock_minutes,

    CASE 
        WHEN sp.loading_dock_scan_at IS NOT NULL 
            AND sp.loaded_scan_at IS NOT NULL
        THEN TIMESTAMPDIFF(MINUTE, sp.loading_dock_scan_at, sp.loaded_scan_at)
        ELSE NULL
    END AS loading_dock_to_loaded_minutes,

-- SCAN TIMESTAMPS
    sp.inbound_scan_at,
    sp.dimensioning_scan_at,
    sp.destination_scan_at,
    sp.loading_dock_scan_at,
    sp.loaded_scan_at,

-- SCAN RESULTS MESSAGE
    sp.inbound_scan_result,
    sp.dimensioning_scan_result,
    sp.destination_scan_result,
    sp.loading_dock_scan_result,
    sp.loaded_scan_result,

-- SCAN RESULTS FLAG
    sp.inbound_scan_success,
    sp.inbound_scan_failed,
    
    sp.dimensioning_scan_success,
    sp.dimensioning_scan_failed,

    sp.destination_scan_success,
    sp.destination_scan_failed,

    sp.loading_scan_success,
    sp.loading_scan_failed,

-- BUSINESS FLAGS
    CASE
        WHEN sp.dimensioning_scan_result = 'SKIPPED'
            AND sp.destination_scan_at IS NOT NULL
        THEN 1 ELSE 0
    END AS skipped_dimensioning,

    sp.failed_stage,
    sp.is_loaded,
    sp.not_loaded,
    sp.is_exception

FROM parcel_sorting_hub.silver.parcels p

LEFT JOIN parcel_sorting_hub.silver.intake i
    ON p.delivery_id = i.delivery_id

LEFT JOIN parcel_sorting_hub.silver.loading l 
    ON p.loading_id = l.loading_id

LEFT JOIN parcel_sorting_hub.gold.parcel_scan_process sp
    ON p.parcel_id = sp.parcel_id
    
LEFT JOIN parcel_sorting_hub.silver.sorting s
    ON p.parcel_id = s.parcel_id;

# `gold.fact_scan_events`

### Purpose

This fact table keeps scan data at event level.

It is used for detailed scan analysis, device performance, scan failure tracking, and shift-based operational reporting.

### Grain

One row per `scan_id`.

### Source Tables
- silver.scans
- silver.parcels
- silver.devices

### Business Logic

The table includes:

- scan timestamp,
- scan date and hour,
- scan type,
- scan result,
- scan success/failure flags,
- device type and zone,
- dynamic scan weight,
- actual parcel weight,
- weight difference,
- shift period,
- parcel route,
- service type.__

### Power BI Usage

**This table should be used for scan-level analysis, including:**

- scan volume by hour,
- scan failures by device,
- scan failures by zone,
- scan type performance,
- shift-based scan performance,
- dynamic weight vs actual weight analysis.

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.fact_scan_events
USING DELTA 
AS

SELECT
    s.scan_id,
    s.parcel_id,
    p.delivery_id,
    p.loading_id,

    s.device_id,
    d.type AS device_type,
    d.zone AS device_zone,
    s.scan_type,
    s.result,
    s.scan_timestamp,

    DATE(s.scan_timestamp) AS scan_date,
    HOUR(s.scan_timestamp) AS scan_hour,

    CASE WHEN s.result = 'OK' THEN 1 ELSE 0 END AS is_scan_ok,
    CASE WHEN s.result = 'FAIL' THEN 1 ELSE 0 END AS is_scan_fail,

    s.dynamic_weight,
    p.weight_kg AS actual_weight,
    (p.weight_kg - s.dynamic_weight) AS weight_diff_kg,

-- SHIFT CATEGORIZATION
    CASE 
        WHEN HOUR(s.scan_timestamp) BETWEEN 6 AND 13 THEN 'MORNING'
        WHEN HOUR(s.scan_timestamp) BETWEEN 14 AND 21 THEN 'AFTERNOON'
        ELSE 'NIGHT'
    END AS shift_period,

    p.source_hub_id,
    p.destination_hub_id,
    CONCAT('FROM ', p.source_hub_id, ' TO ', p.destination_hub_id) AS parcel_travel_route,
    p.service_type

FROM parcel_sorting_hub.silver.scans s

LEFT JOIN parcel_sorting_hub.silver.parcels p
    ON s.parcel_id = p.parcel_id
    
LEFT JOIN parcel_sorting_hub.silver.devices d  
    ON s.device_id = d.device_id;

# `gold.fact_exceptions`

### Purpose

This fact table stores exception-level operational data.

It is used to analyze parcel exceptions, resolution time, error types, and employee/device involvement.

### Grain

One row per `exception_id`.

### Source Tables
- silver.exceptions

### Business Logic

The table includes:

- exception report date,
- report hour,
- resolve hour,
- parcel ID,
- employee ID,
- device ID,
- error code,
- reported timestamp,
- resolved timestamp,
- time to resolve in minutes,
- resolution-in-time flag.
- Key Metrics Created
- time_to_resolve_minutes
- is_resolved_in_time

### Power BI Usage

**This table should be used for exception reporting, including:**

- exception volume,
- exception rate,
- average resolution time,
- exceptions by error code,
- exceptions by hour,
- exceptions by employee,
- exceptions by device.

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.fact_exceptions 
USING DELTA 
AS 

WITH exceptions_overview AS (
  SELECT
    DATE(reported_at) AS date,
    HOUR(reported_at) AS report_hour,
    HOUR(resolved_at) AS resolve_hour,

-- SHIFT CATEGORIZATION
    CASE 
        WHEN HOUR(reported_at) BETWEEN 6 AND 13 THEN 'MORNING'
        WHEN HOUR(reported_at) BETWEEN 14 AND 21 THEN 'AFTERNOON'
        ELSE 'NIGHT'
    END AS shift_period_reported_at,

    CASE 
        WHEN HOUR(resolved_at) BETWEEN 6 AND 13 THEN 'MORNING'
        WHEN HOUR(resolved_at) BETWEEN 14 AND 21 THEN 'AFTERNOON'
        ELSE 'NIGHT'
    END AS shift_period_resolved_at,

    exception_id,
    parcel_id,
    employee_id,
    device_id,
    error_code,
    reported_at,
    resolved_at,
    TIMESTAMPDIFF(MINUTE, reported_at, resolved_at) AS time_to_resolve_minutes
  FROM parcel_sorting_hub.silver.exceptions
),

exceptions_in_depth AS (
  SELECT
    *,
    CASE 
      WHEN time_to_resolve_minutes <= 150 THEN 1
      ELSE 0
    END AS is_resolved_in_time
  FROM exceptions_overview
)

SELECT
  date,
  report_hour,
  resolve_hour,
  exception_id,
  parcel_id,
  employee_id,
  device_id,
  error_code,
  reported_at,
  resolved_at, 
  shift_period_reported_at,
  shift_period_resolved_at,
  time_to_resolve_minutes,
  is_resolved_in_time
FROM exceptions_in_depth


# Dimension Tables

# `gold.dim_date`

### Purpose

This dimension table provides date attributes for Power BI filtering and time-based analysis.

### Grain

One row per `date`.

### Source Table
- gold.fact_parcel_journey

### Attributes

The table includes:

- date,
- year,
- month number,
- month name,
- day,
- day name,
- week number.

Power BI Usage

**Use this table as the main date dimension for parcel-level reporting.
It should be connected to fact tables using date fields such as:**

- received_date
- scan_date
- exception report date, *if needed*.

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.dim_date
USING DELTA
AS

SELECT DISTINCT
    received_date AS date,
    YEAR(received_date) AS year,
    MONTH(received_date) AS month_number,
    DATE_FORMAT(received_date, 'MMM') AS month_name,
    DAY(received_date) AS day,
    DATE_FORMAT(received_date, 'E') AS day_name,
    WEEKOFYEAR(received_date) AS week_number
FROM parcel_sorting_hub.gold.fact_parcel_journey;

# `gold.dim_hour`

### Purpose

This dimension table provides hour and shift attributes for operational time-of-day analysis.

### Grain

One row per `hour`.

### Source Table
- gold.fact_scan_events

### Business Logic

Hours are grouped into operational shifts:

- 06:00-13:00 -> `MORNING`
- 14:00-21:00 -> `AFTERNOON`
- remaining hours -> `NIGHT`

The table also includes shift_sort_order to make shift ordering easier in Power BI.

### Power BI Usage

Use this table for:

- hourly scan volume,
- hourly failure rates,
- shift-level performance,
- operational workload analysis.

In [0]:
CREATE OR REPLACE TABLE parcel_sorting_hub.gold.dim_hour
USING DELTA
AS

SELECT DISTINCT
    scan_hour AS hour,
    CASE 
        WHEN scan_hour BETWEEN 6 AND 13 THEN 'MORNING'
        WHEN scan_hour BETWEEN 14 AND 21 THEN 'AFTERNOON'
        ELSE 'NIGHT'
    END AS work_shift,
    CASE 
        WHEN scan_hour BETWEEN 6 AND 13 THEN 1
        WHEN scan_hour BETWEEN 14 AND 21 THEN 2
        ELSE 3
    END AS shift_sort_order
FROM parcel_sorting_hub.gold.fact_scan_events;